In [2]:
def _process_value(raw_value) -> float:
  return raw_value / (1024**3)

In [7]:
sdk_output = "['8928295936', '8928215040', '8928215040', '8928215040'] "

cleaned = sdk_output.strip().strip("[]").replace("'", "").replace('"', "")
print(cleaned)
values = []
for x in cleaned.split(","):
      # Use _process_value to convert bytes to GiB
    values.append(_process_value(float(x.strip())))

print(values)
print(len(values))

8928295936, 8928215040, 8928215040, 8928215040
[8.315123558044434, 8.315048217773438, 8.315048217773438, 8.315048217773438]
4


In [9]:
import ast
raw_list = ast.literal_eval(sdk_output)
print(raw_list)
print(type(raw_list))

['8928295936', '8928215040', '8928215040', '8928215040']
<class 'list'>


In [1]:
v1 = 8.32
v2 = 8.315048217773438
print(f"{v1:<15.2f} | {v2:<15.2f}")

8.32            | 8.32           


In [4]:
stdout = "2026-03-04T07:34:38Z   RestartJobSetFailurePolicyAction   applying RestartJobSet failure policy action (first failed job: ttr-pod-delete-exist-20260304072850-tpu-job-slice-0)"
lines = stdout.strip().splitlines()

print(lines)
first_line = lines[0].split()
print(first_line)
time_str = first_line[0]

['2026-03-04T07:34:38Z   RestartJobSetFailurePolicyAction   applying RestartJobSet failure policy action (first failed job: ttr-pod-delete-exist-20260304072850-tpu-job-slice-0)']
['2026-03-04T07:34:38Z', 'RestartJobSetFailurePolicyAction', 'applying', 'RestartJobSet', 'failure', 'policy', 'action', '(first', 'failed', 'job:', 'ttr-pod-delete-exist-20260304072850-tpu-job-slice-0)']


In [13]:
import datetime
dt_object = datetime.datetime.fromisoformat(time_str.replace("Z", "+00:00"))
print(time_str)
print(dt_object)
print(type(time_str))
dt_object

2026-03-04T07:34:38Z
2026-03-04 07:34:38+00:00
<class 'str'>


datetime.datetime(2026, 3, 4, 7, 34, 38, tzinfo=datetime.timezone.utc)

In [8]:
int(dt_object.timestamp())

1772609678

In [ ]:
from datetime import datetime

ts = 1773703620

dt_object = datetime.fromtimestamp(ts)

print(dt_object.strftime("%Y-%m-%d %H:%M:%S"))

2026-03-16 23:27:00


In [2]:
from dataclasses import dataclass
from enum import auto
from enum import IntEnum
import re

In [ ]:
from dataclasses import dataclass
from enum import IntEnum, auto

# A type alias for a parsed row, mapping column headers to their values.
_TableRow = dict[str, str]

@dataclass
class Table:
  """Represents a single parsed table from the tpu-info output."""

  name: str
  raw_body: str
  body: list[_TableRow] | None = None

  def parse_body(self):
    """Parses the raw_body string to populate the structured body attribute."""

    class TableLineIndex(IntEnum):
      """Below is an example of the text returned by tpu-info, formatted as a table.

      | Chip        | Type         | Devices | PID |
      |-------------|--------------|---------|-----|
      | /dev/vfio/0 | TPU v6e chip | 1       | 24  |
      | /dev/vfio/1 | TPU v6e chip | 1       | 24  |
      | /dev/vfio/2 | TPU v6e chip | 1       | 24  |
      | /dev/vfio/3 | TPU v6e chip | 1       | 24  |
      """

      HEADER = 0
      SEPARATOR = 1
      DATA = 2

    lines = self.raw_body.strip().split("\n")

    # 至少要有標題與分隔線才能算是有效的表格
    if len(lines) < TableLineIndex.DATA:
      self.body = []
      return

    header_line = lines[TableLineIndex.HEADER]
    # 將標題用 '|' 切割。使用 [1:-1] 來濾除最左邊與最右邊的空字串（因為 Markdown 首尾都有 '|'）
    headers = [h.strip() for h in header_line.split("|")[1:-1]]

    # 資料行從 DATA 索引開始直到最後
    data_lines = lines[TableLineIndex.DATA :]

    parsed_body = []
    for line in data_lines:
      columns = line.split("|")[1:-1]
      if len(columns) != len(headers):
        continue

      row_data: _TableRow = {
          header: col.strip() for header, col in zip(headers, columns)
      }
      parsed_body.append(row_data)

    self.body = parsed_body

In [ ]:

def parse_tpu_info_output(output: str) -> list[Table]:
  """Splits a multi-table Markdown string from tpu-info into a structured TpuInfo object.

  Args:
    output: The raw string output from the 'tpu-info' command.

  Returns:
    A list of Table objects with attributes populated for each found table.
  """
  # 正規表達式解析：
  # 1. ^([A-Za-z][^\n]*?)\s*\n+  : 捕捉標題 (以英文字母開頭，並過濾掉行尾多餘的空白)
  # 2. (?:^[ \t]*\n)* : 忽略標題與表格中間的純空白/空行
  # 3. (^\|[^\n]*(?:\n\|[^\n]*)*): 捕捉表格本體 (連續以 '|' 開頭的行)
  pattern = re.compile(
      r"^([A-Za-z][^\n]*?)\s*\n+"
      r"(^\|[^\n]*(?:\n\|[^\n]*)*)",
      re.MULTILINE
  )

  parsed_tables = []

  # 使用 finditer 可以確保標題與表格是成對匹配的，同時也會自動忽略中間的 Warning Box (╭───╰)
  for match in pattern.finditer(output):
    name = match.group(1).strip()
    raw_body = match.group(2).strip()

    table = Table(name=name, raw_body=raw_body, body=None)

    # 這裡會呼叫 Table 內部的解析邏輯，請務必確認它支援 Markdown 解析
    table.parse_body()
    parsed_tables.append(table)

  if not parsed_tables:
    raise ValueError("Failed to parse any tables from the tpu-info output.")

  return parsed_tables

In [ ]:
full_output = """
TPU Chips

| Chip        | Type         | Devices | PID |
|-------------|--------------|---------|-----|
| /dev/vfio/0 | TPU v6e chip | 1       | 24  |
| /dev/vfio/1 | TPU v6e chip | 1       | 24  |
| /dev/vfio/2 | TPU v6e chip | 1       | 24  |
| /dev/vfio/3 | TPU v6e chip | 1       | 24  |

TPU Runtime Utilization

| Chip | HBM Usage (GiB)      | Duty cycle |
|------|----------------------|------------|
| 0    | 8.44 GiB / 31.25 GiB | 100.00%    |
| 1    | 8.44 GiB / 31.25 GiB | 100.00%    |
| 4    | 8.44 GiB / 31.25 GiB | 100.00%    |
| 5    | 8.44 GiB / 31.25 GiB | 100.00%    |

TensorCore Utilization

| Core ID | TensorCore Utilization |
|---------|------------------------|
| 0       | 8.71%                  |
| 1       | 8.56%                  |
| 2       | 8.52%                  |
| 3       | 8.20%                  |

TPU Buffer Transfer Latency

| Buffer Size | P50         | P90         | P95          | P999         |
|-------------|-------------|-------------|--------------|--------------|
| 8MB+        | 54031.13 us | 96579.79 us | 103413.24 us | 131976.57 us |

TPU Inbound Buffer Transfer Latency

| Buffer Size | P50         | P90         | P95          | P999         |
|-------------|-------------|-------------|--------------|--------------|
| 8MB+        | 53198.63 us | 93329.45 us | 101954.57 us | 132554.97 us |

╭──────────────────────── Host Compute Latency Status ─────────────────────────╮
│ WARNING: Host Compute Latency metrics unavailable. Did you start a           │
│ MULTI_SLICE workload with `TPU_RUNTIME_METRICS_PORTS=8431,8432,8433,8434`?   │
╰──────────────────────────────────────────────────────────────────────────────╯
TPU gRPC TCP Minimum RTT

| P50      | P90      | P95      | P999     |
|----------|----------|----------|----------|
| 66.79 us | 82.26 us | 84.50 us | 86.18 us |

TPU gRPC TCP Delivery Rate

| P50           | P90           | P95           | P999          |
|---------------|---------------|---------------|---------------|
| 12408.00 Mbps | 27133.21 Mbps | 30492.25 Mbps | 35873.24 Mbps |
"""

tpu_info_output = parse_tpu_info_output(full_output)
print(tpu_info_output)
for i in tpu_info_output:
  print(i.name)
  print(i.body)

[Table(name='TPU Chips', raw_body='| Chip        | Type         | Devices | PID |\n|-------------|--------------|---------|-----|\n| /dev/vfio/0 | TPU v6e chip | 1       | 24  |\n| /dev/vfio/1 | TPU v6e chip | 1       | 24  |\n| /dev/vfio/2 | TPU v6e chip | 1       | 24  |\n| /dev/vfio/3 | TPU v6e chip | 1       | 24  |', body=[{'Chip': '/dev/vfio/0', 'Type': 'TPU v6e chip', 'Devices': '1', 'PID': '24'}, {'Chip': '/dev/vfio/1', 'Type': 'TPU v6e chip', 'Devices': '1', 'PID': '24'}, {'Chip': '/dev/vfio/2', 'Type': 'TPU v6e chip', 'Devices': '1', 'PID': '24'}, {'Chip': '/dev/vfio/3', 'Type': 'TPU v6e chip', 'Devices': '1', 'PID': '24'}]), Table(name='TPU Runtime Utilization', raw_body='| Chip | HBM Usage (GiB)      | Duty cycle |\n|------|----------------------|------------|\n| 0    | 8.44 GiB / 31.25 GiB | 100.00%    |\n| 1    | 8.44 GiB / 31.25 GiB | 100.00%    |\n| 4    | 8.44 GiB / 31.25 GiB | 100.00%    |\n| 5    | 8.44 GiB / 31.25 GiB | 100.00%    |', body=[{'Chip': '0', 'HBM Usage 

In [ ]:
import sys
import os
root_path = os.path.abspath(os.path.join(os.getcwd(), "../../"))

if root_path not in sys.path:
    sys.path.append(root_path)
from datetime import timedelta

from dags.tpu_observability.utils.gcp_util import list_time_series, query_time_series
from dags.tpu_observability.utils.time_util import TimeUtil

start_time = None
query_start = (
      start_time if start_time else TimeUtil.now() - timedelta(minutes=60)
  )

filter_string = [
    'metric.type="prometheus.googleapis.com/kube_jobset_active_replicas/gauge"',
    'resource.type="prometheus_target"',
    'resource.labels.cluster="yuna-automation"',
    'metric.labels.jobset_name="tpu-info-v6e-workload"',
]

time_series = list_time_series(
    project_id="cienet-cmcs",
    filter_str=" AND ".join(filter_string),
    start_time=query_start,
    end_time=TimeUtil.now(),
  )

print(time_series)

TimeUtil(time=1777515569)
[metric {
  labels {
    key: "replicated_job_name"
    value: "tpu-job-jax-v6e-slice"
  }
  labels {
    key: "jobset_name"
    value: "tpu-info-v6e-workload"
  }
  labels {
    key: "customresource_version"
    value: "v1alpha2"
  }
  labels {
    key: "customresource_kind"
    value: "JobSet"
  }
  labels {
    key: "customresource_group"
    value: "jobset.x-k8s.io"
  }
  type: "prometheus.googleapis.com/kube_jobset_active_replicas/gauge"
}
resource {
  type: "prometheus_target"
  labels {
    key: "project_id"
    value: "cienet-cmcs"
  }
  labels {
    key: "namespace"
    value: "default"
  }
  labels {
    key: "location"
    value: "us-central1"
  }
  labels {
    key: "job"
    value: "kube-state-metrics"
  }
  labels {
    key: "instance"
    value: "kube-state-metrics-0:k8s-objects"
  }
  labels {
    key: "cluster"
    value: "yuna-automation"
  }
}
metric_kind: GAUGE
value_type: DOUBLE
points {
  interval {
    start_time {
      seconds: 1777519

In [ ]:
import sys
import os
root_path = os.path.abspath(os.path.join(os.getcwd(), "../../"))

if root_path not in sys.path:
    sys.path.append(root_path)
from datetime import timedelta

from dags.tpu_observability.utils.gcp_util import list_time_series, query_time_series
from dags.tpu_observability.utils.time_util import TimeUtil
import textwrap

start_time = TimeUtil.now() - timedelta(minutes=10)
end_time = TimeUtil.now()

query = textwrap.dedent(
        f"""
        fetch k8s_container
        | metric 'kubernetes.io/container/multislice/network/dcn_transfer_latencies'
        | filter (
            resource.cluster_name == 'yuna-automation'
            && resource.pod_name == 'tpu-info-v6e-workload-tpu-job-jax-v6e-slice-1-1-f24cd'
        )
        | within {start_time.to_mql_string()}, {end_time.to_mql_string()}
        | align delta(1m)
        | every 1m
        | group_by [resource.pod_name, metric.buffer_size],
            [P50: percentile(val(), 50),
        P90: percentile(val(), 90),
        P95: percentile(val(), 95),
        P999: percentile(val(), 99.9)]
        """
    ).strip()
print(query)
time_series = query_time_series("cienet-cmcs", query)
print(time_series)

fetch k8s_container
| metric 'kubernetes.io/container/multislice/network/dcn_transfer_latencies'
| filter (
    resource.cluster_name == 'yuna-automation'
    && resource.pod_name == 'tpu-info-v6e-workload-tpu-job-jax-v6e-slice-0-1-v78qn'
)
| within d'2026/05/05-02:02:36', d'2026/05/05-02:12:36'
| align delta(1m)
| every 1m
| group_by [resource.pod_name, metric.buffer_size],
    [P50: percentile(val(), 50),
P90: percentile(val(), 90),
P95: percentile(val(), 95),
P999: percentile(val(), 99.9)]
[label_values {
  string_value: "tpu-info-v6e-workload-tpu-job-jax-v6e-slice-0-1-v78qn"
}
label_values {
  string_value: "8MB+"
}
point_data {
  values {
    double_value: 20303.056235485394
  }
  values {
    double_value: 32600.851772696391
  }
  values {
    double_value: 34918.688435414966
  }
  values {
    double_value: 45788.268100893183
  }
  time_interval {
    start_time {
      seconds: 1777947096
    }
    end_time {
      seconds: 1777947096
    }
  }
}
point_data {
  values {
    dou

In [9]:
# 標籤定義
percentiles = ["P50", "P90", "P95", "P99"]

print(f"{'Time (UTC)':<20} | {'Metric':<5} | {'Value':<12}")
print("-" * 45)

for point in time_series[0].point_data:
  time_obj = TimeUtil.from_datetime(point.time_interval.start_time)
  time_str = time_obj.to_iso_string()

  for i, val in enumerate(point.values):
    label = percentiles[i] if i < len(percentiles) else f"V{i}"
    value = val.double_value

    print(f"{time_str:<20} | {label:<5} | {value:>12.2f}")

  print("-" * 45)

Time (UTC)           | Metric | Value       
---------------------------------------------
2026-05-05T02:11:36Z | P50   |     20303.06
2026-05-05T02:11:36Z | P90   |     32600.85
2026-05-05T02:11:36Z | P95   |     34918.69
2026-05-05T02:11:36Z | P99   |     45788.27
---------------------------------------------
2026-05-05T02:10:36Z | P50   |     20329.22
2026-05-05T02:10:36Z | P90   |     32667.68
2026-05-05T02:10:36Z | P95   |     34960.49
2026-05-05T02:10:36Z | P99   |     46753.96
---------------------------------------------
2026-05-05T02:09:36Z | P50   |     20418.92
2026-05-05T02:09:36Z | P90   |     32815.57
2026-05-05T02:09:36Z | P95   |     35052.78
2026-05-05T02:09:36Z | P99   |     47754.68
---------------------------------------------
2026-05-05T02:08:36Z | P50   |     20700.30
2026-05-05T02:08:36Z | P90   |     33071.27
2026-05-05T02:08:36Z | P95   |     35198.55
2026-05-05T02:08:36Z | P99   |     47073.71
---------------------------------------------
2026-05-05T02:07:36Z 

In [2]:
def format_monitoring_time(seconds):
    # 將秒數轉換為 UTC 時間物件
    dt = datetime.fromtimestamp(seconds, tz=timezone.utc)
    # 格式化為易讀字串
    return dt.strftime('%Y-%m-%d %Hㄌ:%M:%S UTC')


start_second_obj = TimeUtil.from_datetime(time_series[0].point_data[0].time_interval.start_time)
end_second_obj = TimeUtil.from_datetime(time_series[0].point_data[0].time_interval.end_time)
print(f"start time: {start_second_obj.to_mql_string()}")
print(f"end time: {end_second_obj.to_mql_string()}")

start time: d'2026/05/04-02:29:22'
end time: d'2026/05/04-02:29:22'
